# Cinéfilo Fuzzy — controlador Mamdani para adequação de filmes

**Mini-Projeto 2 — Sistemas Baseados em Conhecimento (UFPB)**

Continuação do [Cinéfilo](https://github.com/Nicolas-Kleiton/sbc-cinefilo), o Mini-Projeto 1,
que usa regras IF–THEN nítidas em `experta` para montar uma consulta à TMDB.

Aqui o mesmo domínio é atacado por outro lado. O MP1 decide **o que procurar**; este sistema
decide **quão bom é cada candidato encontrado**. A pergunta passa a ser: dado um filme concreto
e uma sessão concreta, qual a adequação dele, de 0 a 10?

A resposta sai de um controlador fuzzy Mamdani com três entradas, defuzzificado por centroide.

## 1. Instalação e importações

In [ ]:
!pip install -q scikit-fuzzy numpy scipy networkx packaging matplotlib requests

In [ ]:
# O scikit-fuzzy 0.5.0 importa networkx e packaging sem declará-los como dependência,
# por isso os dois são instalados explicitamente na célula acima.
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

print("scikit-fuzzy", fuzz.__version__, "| numpy", np.__version__)

## 2. O domínio

Escolher um filme para uma sessão envolve julgamentos que não têm fronteira nítida. Um filme de
115 minutos para quem tem 120 disponíveis "cabe", mas cabe apertado; com 118 minutos ainda cabe,
só que menos. Não existe o instante em que ele deixa de caber — existe uma transição.

Foi exatamente aí que o Mini-Projeto 1 precisou arbitrar. A regra R11 dizia:

> se a duração máxima é 150 minutos ou mais, o piso sobe para 90

O corte em 150 é artificial: uma sessão de 149 minutos e outra de 151 recebem tratamentos
diferentes, embora sejam praticamente a mesma situação. Regras nítidas obrigam a escolher um
ponto; lógica fuzzy deixa a transição ser gradual, que é como a preferência realmente se comporta.

### As três entradas

| Variável | Unidade | O que mede |
|---|---|---|
| `folga` | minutos | tempo disponível menos duração do filme |
| `nota` | 0 a 10 | média das avaliações na TMDB |
| `consenso` | número de votos | quantas pessoas avaliaram o filme |

A `folga` é a variável central, e ela **não é monotônica**: folga negativa é ruim porque o filme
não cabe; folga pequena e positiva é o ideal, porque a sessão é bem aproveitada; folga muito
grande volta a ser menos interessante, porque quem separou três horas e assiste um filme de
setenta minutos subaproveitou o tempo que reservou. Esse "bom no meio, pior nos dois extremos"
é natural em fuzzy e desajeitado em regras nítidas.

O `consenso` existe porque nota e confiança são coisas diferentes. Um filme com 9,0 de média e
40 votos não é comparável a um com 8,5 e 30 mil votos. O MP1 tratou isso com um piso rígido de
votos (`vote_count.gte`, definido pelas regras R5, R6 e R10); aqui a confiança entra como
variável de primeira classe, e um filme com poucos votos não é descartado, apenas pesa menos.

### A saída

`adequacao`, de 0 a 10. É um grau de recomendação, não uma nota de qualidade do filme: o mesmo
filme pode ter adequação alta numa sessão e baixa em outra.

## 3. Variáveis linguísticas e funções de pertinência

Os universos e os pontos de quebra vêm do domínio, não de valores redondos escolhidos por
conveniência. As justificativas estão nos comentários.

In [ ]:
# --------------------------------------------------------------- universos
# folga: o pior caso realista é um filme que estoura a sessão em uma hota e meia;
# do outro lado, 120 min de sobra ja caracterizam "sobra muita".
folga = ctrl.Antecedent(np.arange(-90, 121, 1), "folga")

# nota: escala da propria TMDB.
nota = ctrl.Antecedent(np.arange(0, 10.01, 0.05), "nota")

# consenso: acima de ~3 mil votos a media ja e estavel; o que passa disso nao
# aumenta a confianca, entao o universo satura em 5000 e entradas maiores sao limitadas.
consenso = ctrl.Antecedent(np.arange(0, 5001, 25), "consenso")

adequacao = ctrl.Consequent(np.arange(0, 10.01, 0.05), "adequacao")

# --------------------------------------------------------------- folga
# "estoura": qualquer folga negativa; a 20 min de estouro a pertinencia ja e total,
# porque um filme 20 min mais longo que a sessao inviabiliza a sessao.
folga["estoura"] = fuzz.trapmf(folga.universe, [-90, -90, -20, 0])
# "justa": o filme cabe usando bem o tempo. Pico em 20 min de folga, que cobre
# creditos, pausa e o intervalo natural entre sentar e comecar.
folga["justa"] = fuzz.trimf(folga.universe, [-10, 20, 50])
# "sobra": a partir de ~30 min sobrando; total a partir de 70, quando caberia
# quase outro filme.
folga["sobra"] = fuzz.trapmf(folga.universe, [30, 70, 120, 120])

# --------------------------------------------------------------- nota
# As notas da TMDB concentram-se entre 5 e 8. Abaixo de 5 o filme e mal avaliado
# de fato; acima de 8 esta entre os melhores do catalogo.
nota["fraca"] = fuzz.trapmf(nota.universe, [0, 0, 4, 6])
nota["media"] = fuzz.trimf(nota.universe, [5, 6.5, 8])
nota["alta"] = fuzz.trapmf(nota.universe, [7, 8.5, 10, 10])

# --------------------------------------------------------------- consenso
# Os cortes reaproveitam os patamares que o MP1 ja usava em vote_count.gte:
# 200 para classicos (R10), 500 para sessoes em grupo (R5), 1000 no padrao (R6).
consenso["baixo"] = fuzz.trapmf(consenso.universe, [0, 0, 200, 500])
consenso["medio"] = fuzz.trimf(consenso.universe, [300, 1000, 2500])
consenso["alto"] = fuzz.trapmf(consenso.universe, [1500, 3000, 5000, 5000])

# --------------------------------------------------------------- adequacao
adequacao["baixa"] = fuzz.trapmf(adequacao.universe, [0, 0, 2, 4])
adequacao["media"] = fuzz.trimf(adequacao.universe, [3, 5, 7])
adequacao["alta"] = fuzz.trapmf(adequacao.universe, [6, 8, 10, 10])

# O que faz deste um controlador Mamdani, e nao Sugeno: o consequente e um
# conjunto fuzzy (nao uma funcao das entradas), a implicacao recorta esse
# conjunto pelo minimo, as regras sao agregadas por maximo e a saida nitida sai
# do centroide da area resultante. Os tres ultimos sao o padrao do skfuzzy;
# declarar o centroide explicitamente deixa a escolha visivel no codigo.
adequacao.defuzzify_method = "centroid"

VARIAVEIS = [folga, nota, consenso, adequacao]
for v in VARIAVEIS:
    print(f"{v.label:<10} universo [{v.universe[0]:g}, {v.universe[-1]:g}]  "
          f"termos: {', '.join(v.terms)}")
print(f"\nagregacao: {adequacao.accumulation_method.__name__} | "
      f"defuzzificacao: {adequacao.defuzzify_method}")

In [ ]:
def desenhar(variaveis):
    """Plota as funcoes de pertinencia lado a lado."""
    fig, eixos = plt.subplots(len(variaveis), 1, figsize=(8, 2.1 * len(variaveis)))
    for ax, v in zip(eixos, variaveis):
        for termo in v.terms:
            ax.plot(v.universe, v[termo].mf, linewidth=1.8, label=termo)
        ax.set_title(v.label, loc="left", fontsize=10)
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8, loc="center right")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

desenhar(VARIAVEIS)

## 4. Base de regras

Três entradas com três termos cada geram 27 combinações. Todas as 27 estão na tabela abaixo,
o que torna a cobertura do espaço de entradas verificável por contagem: não existe combinação
de termos sem regra correspondente.

A base segue três princípios:

**O tempo é restrição dura.** Se o filme não cabe na sessão, a adequação é baixa por melhor que
ele seja. As nove regras com `folga = estoura` apontam todas para `baixa`. O que o fuzzy
acrescenta aqui não é nuance no resultado, e sim na transição: estourar por cinco minutos
pertence pouco a "estoura", estourar por uma hora pertence totalmente.

**Nota fraca não se recupera.** Independentemente de caber bem ou de ter muitos votos, um filme
mal avaliado não é recomendado.

**Consenso modula, não decide.** Poucos votos rebaixam a adequação em um nível, nunca mais que
isso. Um filme excelente com pouca votação vira "média", não "baixa".

| folga | nota | consenso baixo | consenso médio | consenso alto |
|---|---|---|---|---|
| estoura | qualquer | baixa | baixa | baixa |
| justa | fraca | baixa | baixa | baixa |
| justa | média | baixa | média | média |
| justa | alta | média | alta | alta |
| sobra | fraca | baixa | baixa | baixa |
| sobra | média | baixa | média | média |
| sobra | alta | média | média | alta |

A diferença entre `justa` e `sobra` aparece numa única célula: nota alta com consenso médio dá
`alta` quando a folga é justa e `média` quando sobra muito tempo. É a penalização por
subaproveitar a sessão — o mesmo julgamento que no MP1 precisou virar o corte rígido de 150
minutos da R11.

In [ ]:
# (folga, nota, consenso) -> adequacao. As 27 combinacoes possiveis, na ordem da tabela.
TABELA = [
    # Nao cabe na sessao: nada mais importa.
    ("estoura", "fraca", "baixo", "baixa"),
    ("estoura", "fraca", "medio", "baixa"),
    ("estoura", "fraca", "alto",  "baixa"),
    ("estoura", "media", "baixo", "baixa"),
    ("estoura", "media", "medio", "baixa"),
    ("estoura", "media", "alto",  "baixa"),
    ("estoura", "alta",  "baixo", "baixa"),
    ("estoura", "alta",  "medio", "baixa"),
    ("estoura", "alta",  "alto",  "baixa"),

    # Cabe aproveitando bem o tempo: o cenario ideal.
    ("justa", "fraca", "baixo", "baixa"),
    ("justa", "fraca", "medio", "baixa"),
    ("justa", "fraca", "alto",  "baixa"),
    ("justa", "media", "baixo", "baixa"),
    ("justa", "media", "medio", "media"),
    ("justa", "media", "alto",  "media"),
    ("justa", "alta",  "baixo", "media"),   # nota alta, mas pouca gente confirmou
    ("justa", "alta",  "medio", "alta"),
    ("justa", "alta",  "alto",  "alta"),

    # Cabe, mas sobra muito tempo: subaproveita a sessao.
    ("sobra", "fraca", "baixo", "baixa"),
    ("sobra", "fraca", "medio", "baixa"),
    ("sobra", "fraca", "alto",  "baixa"),
    ("sobra", "media", "baixo", "baixa"),
    ("sobra", "media", "medio", "media"),
    ("sobra", "media", "alto",  "media"),
    ("sobra", "alta",  "baixo", "media"),
    ("sobra", "alta",  "medio", "media"),   # unica celula que difere de "justa"
    ("sobra", "alta",  "alto",  "alta"),
]

regras = [ctrl.Rule(folga[f] & nota[n] & consenso[c], adequacao[a])
          for f, n, c, a in TABELA]

sistema = ctrl.ControlSystem(regras)
print(f"{len(regras)} regras construidas")

In [ ]:
# Cobertura: as 27 combinacoes de termos existem, cada uma uma unica vez.
from itertools import product

combinacoes = {(f, n, c) for f, n, c, _ in TABELA}
esperadas = set(product(folga.terms, nota.terms, consenso.terms))

assert len(TABELA) == len(combinacoes), "ha combinacao repetida na tabela"
assert combinacoes == esperadas, f"faltam: {esperadas - combinacoes}"
print(f"cobertura completa: {len(esperadas)} combinacoes de termos, {len(TABELA)} regras, "
      "nenhuma lacuna e nenhuma duplicata")

## 5. Inferência e explicação

O `ControlSystemSimulation` faz a inferência e devolve o valor defuzzificado. Para explicar o
resultado, porém, é preciso saber **quais regras participaram e com que força** — o equivalente
fuzzy do trace do Mini-Projeto 1.

A força de ativação de uma regra é o mínimo das pertinências de seus antecedentes, porque os
três termos estão ligados por `&` (conjunção). Calcular isso diretamente, a partir das funções
de pertinência, deixa a explicação transparente e independente da API interna da biblioteca.

In [ ]:
def pertinencia(variavel, termo, valor):
    """Grau com que um valor pertence a um termo linguistico."""
    valor = float(np.clip(valor, variavel.universe[0], variavel.universe[-1]))
    return float(fuzz.interp_membership(variavel.universe, variavel[termo].mf, valor))


def avaliar(v_folga, v_nota, v_consenso):
    """Roda a inferencia e devolve a adequacao defuzzificada.

    Uma simulacao nova por chamada e de proposito: o ControlSystemSimulation do
    skfuzzy mantem cache dos resultados, e reaproveitar a mesma instancia para
    centenas de entradas distintas fica mais lento, nao mais rapido.
    """
    sim = ctrl.ControlSystemSimulation(sistema)
    sim.input["folga"] = float(np.clip(v_folga, -90, 120))
    sim.input["nota"] = float(np.clip(v_nota, 0, 10))
    sim.input["consenso"] = float(np.clip(v_consenso, 0, 5000))
    sim.compute()
    return float(sim.output["adequacao"])


def ativacoes(v_folga, v_nota, v_consenso, minimo=0.01):
    """Forca de ativacao de cada regra, da maior para a menor."""
    saida = []
    for f, n, c, a in TABELA:
        forca = min(pertinencia(folga, f, v_folga),
                    pertinencia(nota, n, v_nota),
                    pertinencia(consenso, c, v_consenso))
        if forca >= minimo:
            saida.append({"forca": forca, "se": (f, n, c), "entao": a})
    return sorted(saida, key=lambda r: -r["forca"])


def explicar(v_folga, v_nota, v_consenso):
    """Mostra a fuzzificacao, as regras ativadas e o resultado."""
    resultado = avaliar(v_folga, v_nota, v_consenso)
    print(f"ENTRADAS: folga={v_folga:+g} min | nota={v_nota:g} | consenso={v_consenso:g} votos")
    print("-" * 76)
    print("Fuzzificacao (grau de pertinencia em cada termo):")
    for variavel, valor in ((folga, v_folga), (nota, v_nota), (consenso, v_consenso)):
        graus = [f"{t}={pertinencia(variavel, t, valor):.2f}" for t in variavel.terms]
        print(f"  {variavel.label:<9} {'  '.join(graus)}")

    ativas = ativacoes(v_folga, v_nota, v_consenso)
    print(f"\nRegras ativadas: {len(ativas)} de {len(TABELA)}")
    for r in ativas:
        f, n, c = r["se"]
        print(f"  forca {r['forca']:.2f}  SE folga={f}, nota={n}, consenso={c} "
              f"ENTAO adequacao={r['entao']}")

    # Mamdani agrega por maximo: quando varias regras concluem o mesmo termo,
    # o grau final daquele termo e a maior das forcas, nunca a soma delas.
    agregado = {}
    for r in ativas:
        agregado[r["entao"]] = max(agregado.get(r["entao"], 0.0), r["forca"])
    resumo = ", ".join(f"{k} ({v:.2f})" for k, v in
                       sorted(agregado.items(), key=lambda kv: -kv[1]))
    print(f"\nAgregacao (maximo por termo de saida): {resumo}")
    print(f"ADEQUACAO (centroide): {resultado:.2f} de 10")
    return resultado

## 6. Casos de teste

Três cenários escolhidos para isolar o efeito de cada entrada. As faixas esperadas estão nos
`assert`, e a interpretação de cada resultado está comentada.

In [ ]:
# CASO 1 - o cenario ideal.
# Sessao de 2h, filme de 100 min: sobram 20 min, o pico exato de "justa".
# Nota 8.4 e 12 mil votos: bem avaliado e com consenso solido.
# Esperado: adequacao alta, acima de 8. Regra dominante: justa + alta + alto -> alta.
r1 = explicar(v_folga=20, v_nota=8.4, v_consenso=12000)
assert r1 > 8.0, r1
print("\nCaso 1 conforme o esperado: o filme aproveita bem a sessao e tem qualidade confirmada.")

In [ ]:
# CASO 2 - o tempo manda.
# Mesmo filme excelente do caso 1, mas agora a sessao tem so 80 min e ele dura 140:
# estoura em 60 min. Nota e consenso continuam otimos.
# Esperado: adequacao baixa, abaixo de 3. A qualidade nao compensa nao caber.
r2 = explicar(v_folga=-60, v_nota=8.4, v_consenso=12000)
assert r2 < 3.0, r2
assert r2 < r1, (r1, r2)
print("\nCaso 2 conforme o esperado: um filme otimo que nao cabe na sessao nao e recomendado.")

In [ ]:
# CASO 3 - consenso fragil.
# Folga identica a do caso 1 e nota ainda mais alta (8.8), mas apenas 90 votos.
# Esperado: adequacao intermediaria, entre 3.5 e 6.5 - nem alta como o caso 1,
# nem baixa. A regra justa + alta + baixo -> media e a dominante.
r3 = explicar(v_folga=20, v_nota=8.8, v_consenso=90)
assert 3.5 < r3 < 6.5, r3
assert r3 < r1, (r1, r3)
print("\nCaso 3 conforme o esperado: nota alta com poucos votos vale menos que nota alta confirmada.")

In [ ]:
# Um caso deliberadamente ambiguo, para ver a interpolacao acontecer.
# folga=35 fica entre "justa" e "sobra"; nota=7.2 fica entre "media" e "alta".
# Varias regras disparam com forcas diferentes e o centroide concilia todas.
explicar(v_folga=35, v_nota=7.2, v_consenso=800)

Nos três casos de teste apenas uma regra dispara, porque as entradas caem no pico dos termos.
O caso ambíguo acima mostra o comportamento que distingue o controlador fuzzy: com `folga` entre
"justa" e "sobra" e `nota` entre "média" e "alta", quatro regras disparam com forças diferentes e
a saída é a conciliação de todas, não a escolha de uma.

Vale notar também que a adequação nunca chega a 0 nem a 10. O centroide de um termo trapezoidal
fica no interior dele: mesmo com a regra `alta` ativada totalmente, o centro de massa da área
resultante fica em torno de 8,4. Isso é característico da defuzzificação por centroide, e não um
defeito da base de regras — a escala útil do controlador é aproximadamente de 1,5 a 8,5.

In [ ]:
# Verificacoes globais sobre o comportamento do controlador.

# 1) Extremos plausiveis. O centroide nao alcanca as bordas do universo:
# com "alta" totalmente ativada a area resultante tem centro de massa em ~8.4.
melhor = avaliar(20, 10.0, 5000)
pior = avaliar(-90, 0.0, 0)
assert melhor > 8.3, melhor
assert pior < 2.0, pior

# 2) Monotonicidade na nota: com folga e consenso fixos, nota maior nunca piora a adequacao.
serie = [avaliar(20, n, 3000) for n in np.arange(0, 10.01, 0.5)]
assert all(b >= a - 1e-6 for a, b in zip(serie, serie[1:])), "adequacao caiu com nota maior"

# 3) Sem lacunas na pratica: nenhuma combinacao de entradas deixa o sistema sem regra ativa.
from itertools import product as _p
vazias = [(f, n, c) for f, n, c in _p(range(-90, 121, 10), np.arange(0, 10.1, 1), range(0, 5001, 250))
          if not ativacoes(f, n, c, minimo=1e-9)]
assert not vazias, f"{len(vazias)} combinacoes sem regra ativa, ex.: {vazias[:3]}"

print(f"melhor cenario: {melhor:.2f} | pior cenario: {pior:.2f}")
print("monotonicidade na nota: ok")
print("nenhuma das combinacoes varridas ficou sem regra ativa")

## 7. Superfície de controle

A superfície mostra como a adequação responde às entradas. É a forma mais direta de ver que a
transição é suave onde o Mini-Projeto 1 tinha degraus.

In [ ]:
def superficie(v_consenso, passo_folga=6, passo_nota=0.5):
    """Adequacao em funcao de folga e nota, com consenso fixo."""
    eixo_f = np.arange(-90, 121, passo_folga)
    eixo_n = np.arange(0, 10.01, passo_nota)
    Z = np.array([[avaliar(f, n, v_consenso) for f in eixo_f] for n in eixo_n])
    return eixo_f, eixo_n, Z

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
for ax, votos in zip(eixos, (150, 5000)):
    ef, en, Z = superficie(votos)
    m = ax.pcolormesh(ef, en, Z, shading="auto")
    ax.set_xlabel("folga (min)")
    ax.set_ylabel("nota")
    ax.set_title(f"consenso = {votos} votos", fontsize=10)
    fig.colorbar(m, ax=ax, label="adequacao")
plt.tight_layout()
plt.show()

print("A faixa clara em torno de folga 0-50 e a regiao ideal. A esquerda dela a adequacao")
print("desaba (o filme nao cabe); a direita ela recua de leve (a sessao e subaproveitada).")
print("Com poucos votos, o grafico inteiro e mais escuro: o consenso rebaixa sem zerar.")

## 8. Pontuando filmes reais da TMDB

Esta seção liga os dois mini-projetos. O Cinéfilo original monta uma consulta a partir das
regras nítidas; aqui os candidatos que a TMDB devolve são pontuados pelo controlador fuzzy e
reordenados por adequação.

A chave da TMDB é opcional: sem ela, tudo acima continua funcionando e apenas esta seção é
pulada.

In [ ]:
import os, requests

TMDB_URL = "https://api.themoviedb.org/3/discover/movie"
DETALHE_URL = "https://api.themoviedb.org/3/movie/{}"

def obter_chave():
    try:
        from google.colab import userdata
        chave = userdata.get("TMDB_API_KEY")
        if chave:
            return chave.strip()
    except Exception:
        pass
    if os.environ.get("TMDB_API_KEY"):
        return os.environ["TMDB_API_KEY"].strip()
    from getpass import getpass
    return getpass("Chave da TMDB (Enter para pular): ").strip()

TMDB_CHAVE = obter_chave()
print("Chave carregada." if TMDB_CHAVE else "Sem chave: esta secao sera pulada.")

def _autenticar(params=None):
    params, headers = dict(params or {}), {"accept": "application/json"}
    if TMDB_CHAVE.startswith("eyJ"):
        headers["Authorization"] = f"Bearer {TMDB_CHAVE}"
    else:
        params["api_key"] = TMDB_CHAVE
    return params, headers

def _get(url, params=None):
    if not TMDB_CHAVE:
        return None
    p, h = _autenticar(params)
    try:
        resp = requests.get(url, params=p, headers=h, timeout=15)
    except requests.RequestException as erro:
        print(f"[TMDB] falha de conexao: {type(erro).__name__}")
        return None
    if resp.status_code != 200:
        print(f"[TMDB] erro HTTP {resp.status_code}")
        return None
    return resp.json()

def candidatos(genero, n=8):
    """Busca filmes de um genero, com os campos que o controlador precisa."""
    dados = _get(TMDB_URL, {"language": "pt-BR", "include_adult": "false",
                            "with_genres": str(genero), "sort_by": "popularity.desc"})
    if not dados:
        return []
    filmes = []
    for f in dados.get("results", [])[:n]:
        detalhe = _get(DETALHE_URL.format(f["id"]))
        if detalhe and detalhe.get("runtime"):
            f["runtime"] = detalhe["runtime"]
            filmes.append(f)
    return filmes

In [ ]:
def ranquear(filmes, tempo_disponivel):
    """Pontua cada filme pelo controlador fuzzy e ordena por adequacao."""
    if not filmes:
        print("Nenhum filme para pontuar (sem chave da TMDB ou busca vazia).")
        return []
    pontuados = []
    for f in filmes:
        folga_min = tempo_disponivel - f["runtime"]
        nota_f = f.get("vote_average", 0)
        votos = f.get("vote_count", 0)
        pontuados.append((avaliar(folga_min, nota_f, votos), folga_min, f))
    pontuados.sort(key=lambda t: -t[0])

    print(f"Sessao de {tempo_disponivel} min. {len(pontuados)} candidatos, "
          "ordenados por adequacao:\n")
    for i, (score, folga_min, f) in enumerate(pontuados, 1):
        ano = (f.get("release_date") or "????")[:4]
        print(f"{i}. {score:5.2f}  {f['title']} ({ano})")
        print(f"          {f['runtime']} min, folga {folga_min:+d} | "
              f"nota {f.get('vote_average', 0):.1f} | {f.get('vote_count', 0)} votos")
    return pontuados

# Genero 16 = Animacao na TMDB. Sessao curta, de hora e meia.
resultado = ranquear(candidatos(16), tempo_disponivel=90)

## 9. Fuzzy e regras nítidas, lado a lado

Os dois mini-projetos resolvem o mesmo domínio com ferramentas diferentes, e a comparação não
é sobre qual é melhor — é sobre o que cada uma pede em troca.

| | Mini-Projeto 1 (experta) | Mini-Projeto 2 (scikit-fuzzy) |
|---|---|---|
| Pergunta respondida | o que procurar | quão bom é cada candidato |
| Entradas | categóricas e numéricas | só numéricas |
| Saída | consulta estruturada | um número de 0 a 10 |
| Encadeamento | 3 níveis, uma regra alimenta outra | nenhum, todas as regras são avaliadas juntas |
| Regras que disparam | subconjunto, resolvido por salience e NOT | todas, com pesos diferentes |
| Conflito entre regras | precisa ser resolvido explicitamente | não existe: tudo é agregado |
| Fronteiras | degraus, definidos por cortes | transições contínuas |
| Explicação | cadeia causal: "porque R1, R3 e R7 dispararam" | distribuição de forças sobre as regras |

**Onde o fuzzy ganha.** Grandezas contínuas com preferência gradual. A R11 do MP1 precisou de um
corte em 150 minutos; aqui a mesma ideia virou o desenho de `folga`, sem nenhuma fronteira
arbitrária. Além disso, não há conflito a resolver: as 27 regras contribuem simultaneamente e o
centroide concilia. No MP1 foi preciso `salience` e `NOT` justamente para impedir que duas
regras concluíssem coisas incompatíveis.

**Onde as regras nítidas ganham.** Tudo que é categórico. "Se há crianças, proibir terror" não
tem meio-termo, e forçar isso num controlador fuzzy seria distorcer o domínio. O encadeamento
também não tem equivalente: no MP1, a R8 usa o que a R1 concluiu, e a R12 usa o que a R8
concluiu. Em Mamdani todas as regras olham apenas as entradas originais — não existe conclusão
intermediária que sirva de premissa para outra regra.

**Custo de manutenção.** Acrescentar uma entrada ao MP1 é escrever uma regra nova. Aqui,
acrescentar uma quarta variável multiplicaria a tabela por três, levando de 27 a 81 regras.
A base fuzzy cresce exponencialmente com o número de variáveis; a base nítida cresce com o
número de situações que interessam.

A conclusão prática é que os dois se complementam, e é isso que a seção 8 mostra: as regras
nítidas decidem o recorte da busca, e o controlador fuzzy ordena o que voltou.